# Quebec Winter Tire Fulfillment Gap — Analysis

**Scenario:** Sales reps in Quebec report losing garage customers ahead of winter tire season, claiming orders couldn't be fulfilled in time. This notebook walks through confirming whether that's true, sizing it, ruling out simpler explanations, quantifying the business impact, and finding the root cause.

**Approach:** confirm it's real -> benchmark against other regions -> rule out the easy explanation -> quantify impact -> find the mechanism.

## Setup

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 50)

conn = sqlite3.connect('../data/tire_distributor.db')

def q(sql, params=None):
    return pd.read_sql_query(sql, conn, params=params)

print('Tables:', q("SELECT name FROM sqlite_master WHERE type='table'")['name'].tolist())

## 1. Is it happening?

Before diagnosing a cause, confirm the complaint is real. Filtering to winter tires specifically (not all orders) matters here — a company-wide number would dilute a problem that's specific to one tire type.

In [ ]:
winter_rates = q("""
SELECT
    w.province,
    SUM(CASE WHEN o.status = 'Backordered' THEN 1 ELSE 0 END) AS n_backordered,
    COUNT(*) AS n_total,
    ROUND(1.0 * SUM(CASE WHEN o.status = 'Backordered' THEN 1 ELSE 0 END) / COUNT(*), 4) AS backorder_rate
FROM orders o
JOIN warehouses w ON o.warehouse_id = w.warehouse_id
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
WHERE p.tire_type = 'Winter'
GROUP BY w.province
ORDER BY backorder_rate DESC
""")
winter_rates

**Finding:** Quebec's winter tire backorder rate is roughly 4-7x every other province. This confirms the complaint isn't anecdotal — it's a real, measurable pattern.

## 2. Ruling out the easy explanation

Before trusting that rate, check whether Quebec just has more order volume overall — that alone could make raw backorder counts look worse without the rate actually being different.

In [ ]:
order_volume = q("""
SELECT w.province, strftime('%Y', o.order_date) AS year, COUNT(*) AS n_orders
FROM orders o
JOIN warehouses w ON o.warehouse_id = w.warehouse_id
GROUP BY w.province, year
ORDER BY w.province, year
""")
order_volume.pivot(index='province', columns='year', values='n_orders')

**Finding:** Order volume is roughly even across provinces (~700-800/year each). This rules out volume as the explanation — the backorder rate gap is a fulfillment issue, not a byproduct of Quebec simply generating more orders.

## 3. Sizing the pattern over time

Checking whether this is a one-off or a recurring pattern across both years in the dataset.

In [ ]:
qc_monthly = q("""
SELECT strftime('%Y-%m', o.order_date) AS month,
       SUM(CASE WHEN o.status = 'Backordered' THEN 1 ELSE 0 END) AS backordered,
       SUM(CASE WHEN o.status = 'Cancelled' THEN 1 ELSE 0 END) AS cancelled
FROM orders o
JOIN warehouses w ON o.warehouse_id = w.warehouse_id
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
WHERE w.province = 'QC' AND p.tire_type = 'Winter'
GROUP BY month
ORDER BY month
""")
qc_monthly.plot(x='month', y=['backordered','cancelled'], figsize=(10,4), title='QC winter tire backorders/cancellations by month')
plt.xticks(rotation=45)
plt.tight_layout()

**Finding:** The spike is concentrated in Sep-Nov and repeats in both 2024 and 2025 — a structural, recurring issue rather than a one-time disruption.

## 4. Quantifying the business impact

Order counts alone don't tell leadership how much this actually costs. Pricing out the unfulfilled quantity turns this into a dollar figure.

In [ ]:
impact = q("""
SELECT
    SUM((oi.quantity_ordered - oi.quantity_fulfilled) * oi.unit_price) AS at_risk_revenue,
    SUM(oi.quantity_ordered - oi.quantity_fulfilled) AS units_short,
    COUNT(DISTINCT o.order_id) AS affected_orders
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
JOIN warehouses w ON o.warehouse_id = w.warehouse_id
WHERE w.province = 'QC'
  AND p.tire_type = 'Winter'
  AND strftime('%m', o.order_date) IN ('09','10','11')
  AND oi.quantity_fulfilled < oi.quantity_ordered
""")
impact

**Finding:** ~$124,169 in at-risk revenue, 738 units short, across 87 affected orders — Quebec winter tires, Sep-Nov only (checked Aug and Dec separately; together they added under 4% to the two-year total, so the window was kept to Sep-Nov for consistency with the rest of the analysis).

This is a floor, not a ceiling — it only counts orders that were placed and partially unfulfilled. It doesn't capture demand that went straight to a competitor without ever becoming an order here.

## 5. Finding the mechanism

Cross-referencing against weekly warehouse inventory snapshots to see whether this is a supply-side (stock) problem.

In [ ]:
inv_by_province = q("""
SELECT strftime('%Y-%m', s.snapshot_date) AS month, w.province, SUM(s.quantity_on_hand) AS qty
FROM inventory_snapshots s
JOIN warehouses w ON s.warehouse_id = w.warehouse_id
JOIN products p ON s.product_id = p.product_id
WHERE p.tire_type = 'Winter'
GROUP BY month, w.province
ORDER BY month
""")
pivot = inv_by_province.pivot(index='month', columns='province', values='qty')
pivot.plot(figsize=(11,5), title='Winter tire inventory on hand by province, monthly')
plt.xticks(rotation=45)
plt.tight_layout()

**Finding:** Quebec (Montreal DC) is the only region where winter tire stock collapses for four consecutive months (Aug-Nov) each year, dropping to roughly 1,200-1,300 units while every other region holds 6,000-8,500 units through their own equivalent season. This inventory timing gap — not a company-wide capacity constraint — is the root cause of Quebec's elevated backorder rate.

See `../report/Quebec_Winter_Tire_Report.docx` for the full write-up, recommendations, and limitations.